# Quilt-VQA Benchmark 1 Extension (Pillar 1b)

Implements Sub-pillar 1b (Section 2.1 / Table 2, Figure 3): once a judge is
validated against pathologist consensus on the PathOPEN/filtered-PathVQA subset
(see `judge_pathologist_agreement.ipynb`), apply it to score **Quilt-VQA's
open-ended pairs** against Benchmark 1 (Knowledge Interpretation/Deduction,
Visual Grounding) - the same benchmark and criteria PathOPEN and filtered-PathVQA
were scored on.

Quilt-VQA (`wisdomik/Quilt_VQA` on HuggingFace, already cached locally) has 985
rows total, split by `answer_type` into 724 `OPEN` and 261 `CLOSED`. The paper's
Sub-pillar 1b scope is explicitly **"Quilt-VQA's open-ended pairs against
Benchmark 1"** only - this notebook scores the 724 `OPEN` rows and leaves the 261
`CLOSED` rows out of scope (they were not part of the described judge-validated
comparison; a Benchmark-3 extension to them would be a natural but separate
follow-up, not requested here).

Quilt-VQA has no wrong answers, no MCQ, and no case/image-ID scheme like
PathOPEN - each row is a standalone (image, question, answer) triple, so there is
no external image resolver needed here (images are embedded directly in the HF
dataset).

**Prerequisite**: `judge_pathologist_agreement.ipynb` should have already
established judge-vs-pathologist weighted kappa for Benchmark 1 before treating
these scores as meaningful - this notebook does not re-run that validation, it
assumes it has already been done and simply reports the judges' Quilt-VQA scores
alongside a reminder of where to find that kappa.

## Checkpointing

One judge call per row (724 total per judge). Every call is checkpointed
immediately to `checkpoints/{model_key}_quiltvqa.jsonl`, keyed by the row's
position in the filtered `OPEN` dataset. Re-running the scoring cell after an
interruption skips whatever's already checkpointed.


In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())  # so gpu_allocation/judge_models resolve when the CWD is this dir
from gpu_allocation import cuda_visible_devices_for, describe_allocation, max_memory_for

# Which judge(s) this kernel will load. Declared HERE, before torch touches CUDA,
# because CUDA_VISIBLE_DEVICES has no effect once CUDA is initialized - if a torch CUDA
# op has already run in this kernel, restart it.
#
# Both judges run unquantized at bf16 (~64 GB Qwen / ~76 GB InternVL), so each is
# sharded over 3 of the 47.4 GiB A6000s. They are placed in different NUMA islands
# (Qwen 0-2, InternVL 4-6) so they can run concurrently without sharing a PCIe switch.
MODELS_TO_RUN = ["qwenvl", "internvl"]

os.environ["CUDA_VISIBLE_DEVICES"] = cuda_visible_devices_for(*MODELS_TO_RUN)
print(describe_allocation())

In [ ]:
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

from checkpoint import JudgeCheckpoint
from judge_models import JudgeModel
from prompts.benchmarks import BENCHMARK_1, build_benchmark_1_prompt


In [ ]:
OUTPUT_DIR = os.path.join(os.getcwd(), "quiltvqa_output")
CHECKPOINT_DIR = os.path.join(os.getcwd(), "checkpoints")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


## Load Quilt-VQA, filter to OPEN answer_type

In [ ]:
quilt_vqa = load_dataset("wisdomik/Quilt_VQA")["train"]
quilt_vqa_open = quilt_vqa.filter(lambda row: row["answer_type"] == "OPEN")
len(quilt_vqa), len(quilt_vqa_open)


## Load judge models

In [ ]:
# MODELS_TO_RUN is set in the first cell (it has to be, to pick GPUs before CUDA init).
# max_memory_for(...) returns LOGICAL device ids - CUDA_VISIBLE_DEVICES renumbers cards,
# so with "4,5,6" visible torch sees 0,1,2.
judges = {
    key: JudgeModel(key, max_memory=max_memory_for(key, MODELS_TO_RUN))
    for key in MODELS_TO_RUN
}

# Verify each judge loaded as intended BEFORE committing to a multi-hour run.
# device_map="auto" silently offloads to CPU/disk when a model does not fit, which turns
# a long run into a multi-day one - catch it here, not hours in.
import collections

import torch

for key, judge in judges.items():
    devs = collections.Counter(str(p.device) for p in judge.model.parameters())
    offloaded = [d for d in devs if d in ("cpu", "meta", "disk")]
    print(f"{key}: devices={dict(devs)}")
    print(f"    system_role={judge.use_system_role}  images_in_template={judge.images_in_template}"
          f"  thinking={'<think>' in judge.system_prompt}")
    print(f"    OFFLOAD -> {offloaded if offloaded else 'none (good)'}")
    if offloaded:
        raise RuntimeError(f"{key} offloaded to {offloaded} - lower max_memory or add a card")

judges

## Score Benchmark 1 on Quilt-VQA's open-ended pairs

In [ ]:
def run_quiltvqa_scoring(judge: JudgeModel, model_key: str) -> JudgeCheckpoint:
    checkpoint = JudgeCheckpoint(os.path.join(CHECKPOINT_DIR, f"{model_key}_quiltvqa.jsonl"))
    n_scored = n_skipped = n_failed = 0

    for row_idx, row in enumerate(tqdm(quilt_vqa_open, desc=f"{model_key} Quilt-VQA Benchmark1")):
        item_id = row_idx
        if checkpoint.is_done(item_id):
            n_skipped += 1
            continue
        prompt = build_benchmark_1_prompt(row["question"], row["answer"])
        try:
            scores, raw = judge.score(row["image"], prompt, list(BENCHMARK_1["criteria"].keys()))
        except Exception as e:
            n_failed += 1
            print(f"[checkpoint] FAILED item_id={item_id!r}: {e!r} - will retry next run")
            continue
        checkpoint.append({
            "item_id": item_id,
            "question": row["question"],
            "answer": row["answer"],
            "scores": scores,
            "raw_response": raw,
        })
        n_scored += 1

    print(f"{model_key} Quilt-VQA: scored {n_scored} new, {n_skipped} already done, {n_failed} failed this run")
    return checkpoint


quiltvqa_checkpoints = {model_key: run_quiltvqa_scoring(judges[model_key], model_key) for model_key in MODELS_TO_RUN}


## Post-process: assemble the checkpoint into a CSV

Pure re-read of the checkpoint **file** (via a fresh `JudgeCheckpoint(path)`, not
the in-memory `quiltvqa_checkpoints` object from the scoring cell above) - safe to
re-run any time, independent of the scoring cell above, even in a fresh kernel.

In [ ]:
def assemble_quiltvqa_csv(model_key: str) -> pd.DataFrame:
    """Reads checkpoints/{model_key}_quiltvqa.jsonl directly from disk - does NOT
    depend on the `quiltvqa_checkpoints` dict from the scoring cell, so this is
    safe to run standalone in a fresh kernel."""
    checkpoint = JudgeCheckpoint(os.path.join(CHECKPOINT_DIR, f"{model_key}_quiltvqa.jsonl"))
    records = []
    for record in sorted(checkpoint.load_all(), key=lambda r: r["item_id"]):
        records.append({
            "row_uid": record["item_id"],
            "question": record["question"],
            "answer": record["answer"],
            "Knowledge_Interpretation_Deduction": record["scores"].get("Knowledge Interpretation/Deduction"),
            "Visual_Grounding": record["scores"].get("Visual Grounding"),
        })
    return pd.DataFrame.from_records(records)


for model_key in MODELS_TO_RUN:
    out_df = assemble_quiltvqa_csv(model_key)
    out_path = os.path.join(OUTPUT_DIR, f"{model_key}_quiltvqa_benchmark1.csv")
    out_df.to_csv(out_path, index=False)
    print(model_key, "->", out_path, out_df.shape)


## Figure 3 Panel B equivalent: PathOPEN vs. Quilt-VQA, judge-rated Benchmark 1

Reuses the same Mann-Whitney U / rank-biserial approach as
`judge_pathologist_agreement.ipynb`'s PathOPEN-vs-filtered-PathVQA comparison,
substituting Quilt-VQA as the comparator, using each judge's **own** PathOPEN
scores (from `judge_runner_pathopen.ipynb`'s output) as the PathOPEN side - so the
comparison is judge-vs-judge-on-two-datasets, matching the paper's description
("Forest plot, judge-rated Benchmark 1 scores, PathOPEN vs. Quilt-VQA").

In [ ]:
import numpy as np
from scipy.stats import mannwhitneyu

JUDGE_OUTPUT_DIR = os.path.join(os.getcwd(), "judge_output")


def rank_biserial_from_u(u_stat: float, n1: int, n2: int) -> float:
    return 1 - (2 * u_stat) / (n1 * n2)


def compare_distributions(a: pd.Series, b: pd.Series) -> dict:
    a = a.dropna()
    a = a[a != -1]
    b = b.dropna()
    b = b[b != -1]
    if len(a) < 2 or len(b) < 2:
        return {"n_a": len(a), "n_b": len(b), "u_stat": np.nan, "p_value": np.nan, "rank_biserial": np.nan}
    u_stat, p_value = mannwhitneyu(a, b, alternative="two-sided")
    return {"n_a": len(a), "n_b": len(b), "u_stat": u_stat, "p_value": p_value, "rank_biserial": rank_biserial_from_u(u_stat, len(a), len(b))}


forest_rows = []
for model_key in MODELS_TO_RUN:
    quilt_df = pd.read_csv(os.path.join(OUTPUT_DIR, f"{model_key}_quiltvqa_benchmark1.csv"))
    pathopen_path = os.path.join(JUDGE_OUTPUT_DIR, f"evaluator_{model_key}", "pathopen_eval_data.csv")
    if not os.path.exists(pathopen_path):
        print(f"Skipping {model_key}: run judge_runner_pathopen.ipynb first ({pathopen_path} not found)")
        continue
    pathopen_df = pd.read_csv(pathopen_path)

    for criterion_label, quilt_col, pathopen_col in [
        ("Knowledge Interpretation/Deduction", "Knowledge_Interpretation_Deduction", 'Evaluation OE_Correct_Answer_1\n(Benchmark 1)'),
        ("Visual Grounding", "Visual_Grounding", "OE_Correct_Answer_1_VisGround"),
    ]:
        result = compare_distributions(
            pd.to_numeric(pathopen_df[pathopen_col], errors="coerce"),
            pd.to_numeric(quilt_df[quilt_col], errors="coerce"),
        )
        result.update({"judge": model_key, "criterion": criterion_label, "dataset_a": "PathOPEN", "dataset_b": "Quilt-VQA"})
        forest_rows.append(result)

forest_results = pd.DataFrame(forest_rows)
forest_results


In [ ]:
forest_results.to_csv(os.path.join(OUTPUT_DIR, "pathopen_vs_quiltvqa_mannwhitney.csv"), index=False)

## Reminder: this notebook does not establish judge credibility on its own

The judge-vs-pathologist weighted kappa for Benchmark 1 (the actual validation
step Sub-pillar 1b requires before trusting these Quilt-VQA scores) lives in
`judge_pathologist_agreement.ipynb`'s `all_agreement` table, filtered to
`benchmark == 1`. Report both together, as the paper's Figure 3 does (Panel A:
judge-vs-pathologist kappa on the validation subset; Panel B: judge-rated scores,
PathOPEN vs. Quilt-VQA).

## Raw checkpoint files

`checkpoints/{model_key}_quiltvqa.jsonl` retains every judge call (question,
answer, scores, full raw model response) and is never overwritten.
